# Check 5: Use collective state only as a summary

`CollectiveState` compresses two continuous measurements into a descriptive label. It must not select actions, create a desired state, or replace inspection of spacing and trajectories.

The 0.35 / 0.65 cutoffs follow the state regions used by [Tunstrøm et al. (2013)](https://doi.org/10.1371/journal.pcbi.1002915). Those thresholds distinguish dense regions of their experimental observations; they are not universal physical constants. Our measurements are raw per-step outputs at `dt=0.125`, rather than their experimentally smoothed time series. Sensitivity to the chosen cutoffs is therefore informative.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "shiner_model.py").is_file()
)
MODEL_DIR = PROJECT_ROOT
if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

from shiner_model import GoldenShinersModel, classify_collective_state
from checks.check_support import run_model, audit_run

import shiner_model

## Verify definitions and the exact boundaries

The implementation uses strict inequalities. A value exactly on a boundary is assigned `transition` unless another full condition applies. This makes equality cases explicit and prevents an accidental `<=` change from silently moving records between categories.

`polarized`: P > high and R < low; `milling`: P < low and R > high; `swarm`: P < low and R < low; all other combinations: `transition`. Here P is polarization and R is rotation.

In [2]:
boundary_cases = [
    ("Aligned", 0.9, 0.1, "polarized"),
    ("Circular", 0.1, 0.9, "milling"),
    ("Low order", 0.1, 0.1, "swarm"),
    ("Intermediate", 0.5, 0.5, "transition"),
    ("P exactly high", 0.65, 0.1, "transition"),
    ("R exactly low", 0.9, 0.35, "transition"),
    ("P exactly low", 0.35, 0.1, "transition"),
    ("R exactly high", 0.1, 0.65, "transition"),
    ("P just above high", 0.650001, 0.1, "polarized"),
    ("P just below low", 0.349999, 0.1, "swarm"),
]
boundary_results = []
for name, polarization, rotation, expected in boundary_cases:
    actual = classify_collective_state(polarization, rotation)
    assert actual == expected
    boundary_results.append({
        "Case": name, "Polarization": polarization, "Rotation": rotation,
        "Expected": expected, "Actual": actual, "Result": "passed",
    })
pd.DataFrame(boundary_results)

,Case,Polarization,Rotation,Expected,Actual,Result
0,Aligned,0.900000,0.10,polarized,polarized,passed
1,Circular,0.100000,0.90,milling,milling,passed
2,Low order,0.100000,0.10,swarm,swarm,passed
3,Intermediate,0.500000,0.50,transition,transition,passed
4,P exactly high,0.650000,0.10,transition,transition,passed
5,R exactly low,0.900000,0.35,transition,transition,passed
6,P exactly low,0.350000,0.10,transition,transition,passed
7,R exactly high,0.100000,0.65,transition,transition,passed
8,P just above high,0.650001,0.10,polarized,polarized,passed
9,P just below low,0.349999,0.10,swarm,swarm,passed


## State occupancy across ten seeds

We repeat 500-step runs for seeds 0–9 with all social rules enabled. Counts include t=0: 501 records per seed. The state fractions summarize **time records in these simulations**, not independent biological observations or probabilities for a population. Consecutive timesteps are correlated. We also compare each recorded label to a fresh classification of its saved P/R values.

Whole-group rotation is defined around the global centroid. If the distance graph contains several components, high rotation can also be compatible with separated clusters moving around that shared centroid. Use connectivity and spatial trajectories before interpreting the label as one coherent milling school; a rotation trace alone does not establish that.

In [3]:
SEEDS = list(range(10))
STEPS = 500
STATE_ORDER = ["polarized", "milling", "swarm", "transition"]
state_frames = []
for seed in SEEDS:
    case = run_model(n_fish=20, seed=seed, steps=STEPS)
    audit_run(case)
    frame = case.datacollector.get_model_vars_dataframe().copy()
    expected = [classify_collective_state(p, r) for p, r in zip(frame["Polarization"], frame["Rotation"])]
    assert expected == frame["CollectiveState"].tolist()
    frame["Seed"] = seed
    frame["Step"] = frame.index
    state_frames.append(frame)
state_data = pd.concat(state_frames, ignore_index=True)
occupancy = pd.crosstab(state_data["Seed"], state_data["CollectiveState"]).reindex(columns=STATE_ORDER, fill_value=0)
assert occupancy.sum(axis=1).eq(STEPS + 1).all()
occupancy_percent = 100 * occupancy.div(occupancy.sum(axis=1), axis=0)
print(f"{len(state_data)} records classified consistently across {len(SEEDS)} seeds.")
occupancy_percent.round(2)

5010 records classified consistently across 10 seeds.


CollectiveState,polarized,milling,swarm,transition
Seed,,,,
0,2.20,51.3,2.79,43.71
1,82.04,0.0,12.97,4.99
2,46.11,0.0,5.79,48.10
3,72.26,0.0,8.58,19.16
4,45.71,1.2,7.39,45.71
5,77.84,0.0,4.79,17.37
6,76.25,0.0,6.79,16.97
7,82.63,0.0,3.79,13.57
8,73.25,0.0,5.19,21.56


## Sensitivity to the threshold pair

We reclassify exactly the same saved continuous metrics with three threshold pairs. **No simulation is rerun for this comparison.** If labels change while P/R stay fixed, the change reflects the descriptive categorization rather than different fish behaviour.

We include all records, including initial transients, and report percentage points in each state and the percentage of records relabelled relative to the baseline.

The alternative pairs are **0.25 / 0.75** and **0.40 / 0.60**, compared with the baseline **0.35 / 0.65**. These are the pairs examined in Supplementary Figure S3 of [Tunstrøm et al. (2013)](https://doi.org/10.1371/journal.pcbi.1002915); applying them here remains a sensitivity check on our own model outputs.

In [4]:
threshold_pairs = [(0.25, 0.75), (0.35, 0.65), (0.40, 0.60)]
classified_labels = {}
sensitivity_rows = []
baseline_labels = state_data["CollectiveState"].to_numpy()
for low, high in threshold_pairs:
    labels = np.array([
        classify_collective_state(p, r, low=low, high=high)
        for p, r in zip(state_data["Polarization"], state_data["Rotation"])
    ])
    classified_labels[(low, high)] = labels
    counts = pd.Series(labels).value_counts(normalize=True)
    sensitivity_rows.append({
        "Low": low, "High": high,
        **{state: 100 * counts.get(state, 0.0) for state in STATE_ORDER},
        "Relabelled records (%)": 100 * np.mean(labels != baseline_labels),
    })
sensitivity = pd.DataFrame(sensitivity_rows).set_index(["Low", "High"])
assert np.isclose(sensitivity[STATE_ORDER].sum(axis=1), 100).all()
assert sensitivity.loc[(0.35, 0.65), "Relabelled records (%)"] == 0
sensitivity.round(2)

,,polarized,milling,swarm,transition,Relabelled records (%)
Low,High,,,,,
0.25,0.75,49.54,4.13,3.85,42.48,11.84
0.35,0.65,57.29,5.33,6.75,30.64,0.00
0.40,0.60,60.12,6.55,8.02,25.31,5.33


## Show records whose labels are sensitive

The records below keep their same seed, timestep, polarization and rotation while threshold choices change. This is why state labels are useful for a compact description but cannot replace the continuous order parameters.

In [5]:
changed = np.zeros(len(state_data), dtype=bool)
for labels in classified_labels.values():
    changed |= labels != baseline_labels
examples = state_data.loc[changed, ["Seed", "Step", "Polarization", "Rotation"]].head(10).copy()
for pair, labels in classified_labels.items():
    examples[f"low={pair[0]:.2f}, high={pair[1]:.2f}"] = labels[examples.index]
examples.round(6)

,Seed,Step,Polarization,Rotation,"low=0.25, high=0.75","low=0.35, high=0.65","low=0.40, high=0.60"
4,0,4,0.259697,0.179762,transition,swarm,swarm
5,0,5,0.313764,0.198288,transition,swarm,swarm
6,0,6,0.341434,0.187722,transition,swarm,swarm
7,0,7,0.321278,0.123521,transition,swarm,swarm
8,0,8,0.325647,0.105630,transition,swarm,swarm
9,0,9,0.304857,0.095636,transition,swarm,swarm
10,0,10,0.374268,0.138779,transition,transition,swarm
11,0,11,0.279461,0.196437,transition,swarm,swarm
12,0,12,0.379706,0.233218,transition,transition,swarm
13,0,13,0.301067,0.180787,transition,swarm,swarm


## Confirm that labels do not influence decisions

We temporarily replace the classifier with a constant label and rerun the same seeded simulation. Numeric measurements and every fish record must remain identical. The original classifier is restored even if an assertion fails. This checks separation between model behaviour and observational summaries.

In [6]:
reference = run_model(n_fish=20, seed=123, steps=100)
original_classifier = shiner_model.classify_collective_state
try:
    shiner_model.classify_collective_state = lambda polarization, rotation, **kwargs: "summary_only"
    relabelled = run_model(n_fish=20, seed=123, steps=100)
finally:
    shiner_model.classify_collective_state = original_classifier

reference_group = reference.datacollector.get_model_vars_dataframe()
relabelled_group = relabelled.datacollector.get_model_vars_dataframe()
pd.testing.assert_frame_equal(
    reference.datacollector.get_agent_vars_dataframe(),
    relabelled.datacollector.get_agent_vars_dataframe(),
)
pd.testing.assert_frame_equal(
    reference_group.drop(columns="CollectiveState"),
    relabelled_group.drop(columns="CollectiveState"),
)
assert relabelled_group["CollectiveState"].eq("summary_only").all()
print("Changing only the summary label left every fish trajectory and numeric metric unchanged.")

Changing only the summary label left every fish trajectory and numeric metric unchanged.


In [7]:
pooled = 100 * state_data["CollectiveState"].value_counts(normalize=True).reindex(STATE_ORDER, fill_value=0)
max_relabelled = sensitivity["Relabelled records (%)"].max()
display(Markdown(f"""## What these outputs tell us

- With the baseline cutoffs, pooled occupancy is **{pooled['polarized']:.1f}% polarized**, **{pooled['milling']:.1f}% milling**, **{pooled['swarm']:.1f}% swarm** and **{pooled['transition']:.1f}% transition** across these {len(state_data)} correlated records.
- Changing the cutoff pair relabels up to **{max_relabelled:.1f}%** of records without changing any trajectory or continuous measurement. The examples above make that dependence visible.
- All ten boundary cases passed, and recorded labels agree with their stored P/R values.
- Replacing the classifier left trajectories and numeric metrics exactly unchanged. The label is therefore an observer-only summary in this implementation.

**Interpretation:** a `swarm` label means low polarization and low rotation under the chosen cutoffs. It does not by itself establish cohesion, density, a stable attractor, or biological validity. A `polarized` label similarly needs spacing and trajectory diagnostics. Any claim about persistent milling should be supported by the actual rotation time series, not just by the existence of a `milling` category.
"""))

## What these outputs tell us

- With the baseline cutoffs, pooled occupancy is **57.3% polarized**, **5.3% milling**, **6.7% swarm** and **30.6% transition** across these 5010 correlated records.
- Changing the cutoff pair relabels up to **11.8%** of records without changing any trajectory or continuous measurement. The examples above make that dependence visible.
- All ten boundary cases passed, and recorded labels agree with their stored P/R values.
- Replacing the classifier left trajectories and numeric metrics exactly unchanged. The label is therefore an observer-only summary in this implementation.

**Interpretation:** a `swarm` label means low polarization and low rotation under the chosen cutoffs. It does not by itself establish cohesion, density, a stable attractor, or biological validity. A `polarized` label similarly needs spacing and trajectory diagnostics. Any claim about persistent milling should be supported by the actual rotation time series, not just by the existence of a `milling` category.
